# IL3.2: Análisis de Trazabilidad y Logs
## Notebook 4: Auditoría, compliance y privacidad en agentes LLM Reales

### Objetivo:
Comprender cómo diseñar sistemas de trazabilidad y logs que cumplan con normativas de privacidad de datos (como GDPR o la Ley de Protección de Datos Personales) y auditoría corporativa ("audit trail"). El estudiante aprenderá a resguardar la privacidad redactando información sensible (PII) e implementando un sistema de hashing para validación de entradas.

### Conceptos Clave:
1. **Audit Trail (Pista de Auditoría):** Registro inmutable de quién hizo qué, cuándo y bajo qué criterios. Es usado para cumplimiento legal.
2. **PII (Personally Identifiable Information):** Datos sensibles de usuarios (correos, identificadores, números de teléfono) que **nunca** deben almacenarse en logs planos.
3. **Redacción de datos:** Enmascarar información privada antes de escribir el log o pasar los datos al LLM.
4. **Hash de entradas:** Generar un identificador criptográfico único (SHA-256) del prompt del usuario. Permite saber si el usuario consultó por algo repetitivo sin necesidad de almacenar el texto original con datos personales en la base de datos de auditoría.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [9]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


✅ LLM de LangChain configurado.
✅ Agente y herramientas listos.


### Sanitización y Auditoría del Agente LLM Real
Para garantizar la privacidad y cumplir con las regulaciones de cumplimiento (compliance), debemos redactar la PII de la consulta del usuario *antes* de enviarla al LLM o al agente de Wikipedia, y guardar solo la consulta redactada junto al hash de la consulta original en nuestro archivo de auditoría `agent_audit.jsonl`.


In [11]:
import hashlib
import json
from datetime import datetime
import re
import uuid
import random
import time

audit_file = "agent_audit.jsonl"

def hash_input(input_text):
    # Hash criptográfico para verificación de integridad sin almacenar el texto sensible
    return hashlib.sha256(input_text.encode("utf-8")).hexdigest()

def redact_sensitive_data(text):
    # Redactar RUTs chilenos (ej. 18.234.567-9)
    text = re.sub(r"\b\d{1,2}(?:\.?\d{3}){2}-[\dkK]\b", "[RUT_REDACTED]", text)
    # Redactar correos electrónicos
    text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", "[EMAIL_REDACTED]", text)
    return text

def write_audit(entry):
    with open(audit_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


### Wrapper Seguro para el Agente Real
Implementaremos `run_secure_wikipedia_agent`. Este wrapper actúa como una compuerta de seguridad:
1. Recibe la consulta sensible.
2. Redacta la información sensible (PII) y calcula el hash de la entrada original.
3. Envía **la versión redactada** al agente real `agent_executor` (protegiendo la privacidad de los datos en APIs externas).
4. Registra la transacción en el log de auditoría inmutable indicando si requiere revisión humana.


In [12]:
def run_secure_wikipedia_agent(user_id, query):
    trace_id = str(uuid.uuid4())
    
    # Pre-procesamiento de Seguridad (Redacción y Hashing)
    redacted_query = redact_sensitive_data(query)
    hashed_input = hash_input(query)
    
    # Determinar si contenía PII originalmente
    had_pii = redacted_query != query
    
    start_time = time.time()
    try:
        if llm is None:
            # Simulación por falta de API keys
            response_text = f"Resultado simulado para consulta redactada: '{redacted_query}'."
        else:
            # Ejecución real del agente con la consulta sanitizada
            response = agent_executor.invoke({"input": redacted_query})
            response_text = response.get("output", "")
    except Exception as e:
        response_text = f"Error en ejecución: {e}"
        
    duration = round(time.time() - start_time, 4)
    
    # Regla de Compliance: derivar a revisión humana si la consulta contenía PII
    # o si la respuesta tardó demasiado tiempo
    needs_human_review = had_pii or (duration > 5.0)
    
    audit_entry = {
        "timestamp": datetime.now().isoformat(),
        "trace_id": trace_id,
        "user_id": user_id,
        "input_hash": hashed_input,
        "redacted_input": redacted_query,
        "response": response_text,
        "response_time": duration,
        "human_review_required": needs_human_review,
        "compliance_checked": True
    }
    
    write_audit(audit_entry)
    return response_text


### Ejecución de Pruebas de Auditoría
Realizaremos consultas con datos sensibles y verificaremos que no se guarden en el archivo de auditoría.


In [13]:
# Limpiar archivo de auditoría
with open(audit_file, "w", encoding="utf-8") as f:
    pass

# Consulta 1: Contiene correo
run_secure_wikipedia_agent("user_abc", "Hola, mi correo es estudiante@correo.cl. ¿Quién fue Ada Lovelace?")

# Consulta 2: Limpia
run_secure_wikipedia_agent("user_xyz", "Busca información sobre la Torre Eiffel")

# Consulta 3: Contiene RUT chileno
run_secure_wikipedia_agent("user_abc", "Soy el titular del RUT 17.502.993-2. ¿Quién descubrió la Penicilina?")

print("Consultas procesadas. Inspeccionando archivo de auditoría...")




> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Ada Lovelace'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)Hubo un problema al buscar información sobre Ada Lovelace. Sin embargo, puedo decirte que Ada Lovelace fue una matemática y escritora británica del siglo XIX, conocida por su trabajo en la máquina analítica de Charles Babbage. Es considerada la primera programadora de computadoras debido a que escribió el primer algoritmo destinado a ser procesado por una máquina. ¿Te gustaría que intente buscar más información nuevamente?

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Torre Eiffel'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)Hubo un problema al buscar información sobre la Torre Eiffel. ¿Te gustaría que lo intente nuevamente?

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'des

In [14]:
print("--- Contenido del Log de Auditoría (agent_audit.jsonl) ---")
with open(audit_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            print(line.strip())


--- Contenido del Log de Auditoría (agent_audit.jsonl) ---
{"timestamp": "2026-06-05T23:25:53.580214", "trace_id": "1fc53076-354e-48a3-b02b-56ffcb9649d0", "user_id": "user_abc", "input_hash": "f99bc47444c69fc2e07097b766547248cb0a14810f739847048e40d27f510321", "redacted_input": "Hola, mi correo es [EMAIL_REDACTED]. ¿Quién fue Ada Lovelace?", "response": "Hubo un problema al buscar información sobre Ada Lovelace. Sin embargo, puedo decirte que Ada Lovelace fue una matemática y escritora británica del siglo XIX, conocida por su trabajo en la máquina analítica de Charles Babbage. Es considerada la primera programadora de computadoras debido a que escribió el primer algoritmo destinado a ser procesado por una máquina. ¿Te gustaría que intente buscar más información nuevamente?", "response_time": 3.9705, "human_review_required": true, "compliance_checked": true}
{"timestamp": "2026-06-05T23:25:57.378645", "trace_id": "fb274496-5317-47ab-92a6-f0cdb03fcd21", "user_id": "user_xyz", "input_hash"

### 🛠️ Reto Práctico (Mini-entrega)

**Instrucciones:**
1. Extiende la función `redact_sensitive_data` para detectar y redactar números de teléfono chilenos de 9 dígitos (ej. `+56912345678`, `912345678` o con espacios).
2. Modifica el wrapper `run_secure_wikipedia_agent` para que también evalúe si la **respuesta** del LLM contiene algún RUT o correo que no haya sido filtrado, marcándolo en un campo boolean llamado `leak_detected`.
3. Realiza pruebas con una consulta que contenga un número telefónico y muestra las entradas resultantes del archivo de auditoría.


In [15]:
# Desarrolla tu solución aquí

# 1. Función redact_sensitive_data extendida

# 2. Wrapper con detección de leak en la respuesta

# 3. Pruebas y visualización del archivo de auditoría


### 📝 Preguntas de Análisis
1. **¿Por qué es preferible almacenar el Hash SHA-256 de la entrada en lugar de simplemente borrar la entrada o guardarla como texto plano?**
2. **Si el agente requiere usar Wikipedia, ¿por qué es importante redactar la consulta *antes* de enviarla a la API de Wikipedia y al LLM? (Piensa en fugas de datos hacia terceros).**
3. **¿Qué criterios objetivos (además de la confianza matemática del LLM) justificarían que un agente derive una decisión a una revisión por un ser humano (Human-in-the-loop)?**
